# Май–июль: Комиссия месяц — MPOS_RENT vs OPER_DOC (без НДС) vs Excel

Эталон — отчёты эквайринга, колонка **«Комиссия (₽ в месяц)»**. Периметр сверки — строки отчёта, ключ `месяц + ИНН + ID договора`.

| Месяц | Файл |
|---|---|
| 2026-05 | `05_Май_2026.xlsx` |
| 2026-06 | `06_Июнь_2026.xlsx` |
| 2026-07 | `07_Июль_2026.xlsx` |

| Источник | Что берём |
|---|---|
| **MPOS_RENT** | `ods_alpha.scd1_mrc_pos_rent.n_amt` — как шаг 10m `final_df` |
| **OPER_DOC** | `ods.scd1_z_R2_IP_DOG_OPER.c_calc_summ` (это и есть OPER_DOC в озере) |
| **OPER_DOC без НДС** | `round(sum(c_calc_summ) / 1.22, 2)` — с этой суммой сравниваем Excel |

Точное совпадение: `|источник − Excel| ≤ 0.01` ₽. `final_df` эта тетрадка не меняет.

Новый kernel нормален. Нужны Impala и три Excel в `/home/jovyan/documents/Equaring/Data`.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 240)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'qc_oper_doc_vs_mpos_excel_may_jul'
OUT_DIR.mkdir(parents=True, exist_ok=True)

FOCUS_MONTHS = ['2026-05', '2026-06', '2026-07']
PERIOD_START = '2026-05-01'
PERIOD_END = '2026-07-31'
PERIOD_END_EXCL = '2026-08-01'
VAT = 1.22
EPS = 0.01
MEM_LIMIT = '8g'
TOP_N = 30
NOTEBOOK_REV = '2026-10-01-oper-doc-net-v1'

EXCEL_FILES = {
    '2026-05': {'names': ['05_Май_2026.xlsx'], 'header': 0},
    '2026-06': {'names': ['06_Июнь_2026.xlsx'], 'header': 0},
    '2026-07': {'names': ['07_Июль_2026.xlsx'], 'header': -1},
}
MONTHLY_ALIASES = [
    'Комиссия (₽ в месяц)', 'Комиссия (руб в месяц)', 'Комиссия в месяц',
    'Комиссия CN (₽ в месяц)', 'commission_monthly', 'Комиссия \n(₽ в месяц)',
]

print('NOTEBOOK_REV:', NOTEBOOK_REV)
print('FOCUS_MONTHS:', FOCUS_MONTHS)
print('OUT_DIR:', OUT_DIR)
print('MPOS = n_amt | OPER_DOC = round(sum(c_calc_summ) / 1.22, 2)')


## 0) Helpers + Impala


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def to_num(s):
    return pd.to_numeric(
        s.astype(str).str.replace('\xa0', '', regex=False).str.replace(' ', '', regex=False).str.replace(',', '.', regex=False),
        errors='coerce',
    )


def pick_col(columns, aliases):
    cols = list(columns)
    norm = lambda x: re.sub(r'\s+', ' ', str(x).replace('\xa0', ' ').replace('\n', ' ').strip().lower())
    nmap = {norm(c): c for c in cols}
    for a in aliases:
        if a in cols:
            return a
        if norm(a) in nmap:
            return nmap[norm(a)]
    for a in aliases:
        key = norm(a)
        for nk, orig in nmap.items():
            if key and key in nk:
                return orig
    return None


def fetch_imp(sql, label):
    t0 = pd.Timestamp.now()
    print(f'FETCH {label} ...')
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    if df is None:
        df = pd.DataFrame()
    print(f'  rows={len(df):,}  {(pd.Timestamp.now() - t0).total_seconds():.1f}s')
    return df


def is_nz(s):
    return pd.to_numeric(s, errors='coerce').fillna(0).abs() >= EPS


def exact_match(left, right):
    a = pd.to_numeric(left, errors='coerce').fillna(0).round(2)
    b = pd.to_numeric(right, errors='coerce').fillna(0).round(2)
    return np.isclose(a, b, atol=EPS, rtol=0)


def resolve_excel_path(month):
    spec = EXCEL_FILES[month]
    for name in spec['names']:
        p = DATA_DIR / name
        if p.exists():
            return p
    hits = sorted(DATA_DIR.glob(f'*{month[-2:]}*2026*.xlsx'))
    if hits:
        print(f'  fallback Excel {month}: {hits[0].name}')
        return hits[0]
    return None


def read_excel_report(path, header_hint):
    tried = []
    headers = list(range(0, 6)) if header_hint == -1 else [header_hint, *range(0, 6)]
    seen = []
    for h in headers:
        if h in seen:
            continue
        seen.append(h)
        raw = pd.read_excel(path, header=h)
        col_inn = pick_col(raw.columns, ['ИНН', 'inn', 'c_inn'])
        col_agr = pick_col(raw.columns, ['ID договора', 'agr_id', 'abs_agr_id', 'ИД договора'])
        col_mon = pick_col(raw.columns, MONTHLY_ALIASES)
        tried.append((h, col_inn, col_agr, col_mon, list(raw.columns)[:12]))
        if col_inn and col_agr and col_mon:
            return raw, h, col_inn, col_agr, col_mon
    raise RuntimeError(f'Не нашли колонки в {path.name}. tried={tried}')


def classify_row(excel_nz, src_present, src_nz):
    if excel_nz and src_nz:
        return 'оба ≠ 0'
    if (not excel_nz) and (not src_nz):
        return 'оба 0 / нет строки'
    if (not excel_nz) and src_nz:
        return 'источник ≠ 0, отчёт = 0'
    if excel_nz and not src_present:
        return 'отчёт ≠ 0, в источнике нет строки'
    return 'отчёт ≠ 0, источник = 0'


if 'imp' in globals() and imp is not None:
    print('Reuse existing imp')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')


## 1) Эталон: отчёты мая, июня, июля


In [ ]:
excel_parts = []
excel_meta = []
missing = []
for month in FOCUS_MONTHS:
    path = resolve_excel_path(month)
    if path is None:
        missing.append(month)
        print('MISSING Excel for', month, '| DATA_DIR xlsx:', [p.name for p in DATA_DIR.glob('*.xlsx')][:30])
        continue
    header_hint = EXCEL_FILES[month]['header']
    raw_ex, hdr, col_inn, col_agr, col_mon = read_excel_report(path, header_hint)
    col_tar = pick_col(raw_ex.columns, ['Тариф', 'Тарифный план', 'tariff_name'])
    print(f'{month}: {path.name} header={hdr} inn={col_inn} agr={col_agr} monthly={col_mon} rows={len(raw_ex):,}')
    part = pd.DataFrame({
        'report_month': month,
        'inn_key': raw_ex[col_inn].map(normalize_inn_q1),
        'agr_id_key': raw_ex[col_agr].map(normalize_agr_q1),
        'commission_excel': to_num(raw_ex[col_mon]),
        'tariff': raw_ex[col_tar] if col_tar else np.nan,
    })
    part = (
        part.dropna(subset=['agr_id_key'])
        .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
        .agg(commission_excel=('commission_excel', 'max'), tariff=('tariff', 'first'))
    )
    part['commission_excel'] = pd.to_numeric(part['commission_excel'], errors='coerce').fillna(0).round(2)
    part['excel_nz'] = is_nz(part['commission_excel'])
    excel_parts.append(part)
    excel_meta.append({
        'report_month': month,
        'file': path.name,
        'header': hdr,
        'keys': len(part),
        'excel_nz': int(part['excel_nz'].sum()),
        'excel_zero': int((~part['excel_nz']).sum()),
        'sum_excel': float(part['commission_excel'].sum()),
    })

if missing:
    raise FileNotFoundError(
        f'Нет отчётов за {missing}. Положите файлы в {DATA_DIR} '
        f'(05_Май_2026.xlsx, 06_Июнь_2026.xlsx, 07_Июль_2026.xlsx) и перезапустите.'
    )

ex_all = pd.concat(excel_parts, ignore_index=True)
excel_stat = pd.DataFrame(excel_meta)
excel_stat['excel_nz_pct'] = (100.0 * excel_stat['excel_nz'] / excel_stat['keys']).round(2)
print('=== Эталон Excel май–июль ===')
display(excel_stat)
print('keys all months=', f'{len(ex_all):,}', '| unique agr=', f'{ex_all["agr_id_key"].nunique():,}')


## 2) MPOS_RENT: `n_amt` за май–июль


In [ ]:
sql_mpos = f'''
with rent_base as (
  select
    cast(c_nmrc as string) as c_nmrc,
    cast(d_rent as date) as d_rent_dt,
    cast(n_amt as double) as n_amt_num,
    cast(ods_commit_ts as timestamp) as ods_commit_ts,
    cast(ods_insert_ts as timestamp) as ods_insert_ts,
    cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
    coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
  from ods_alpha.scd1_mrc_pos_rent
  where c_nmrc is not null
    and cast(d_rent as date) between cast('{PERIOD_START}' as date) and cast('{PERIOD_END}' as date)
),
rent_ranked as (
  select *,
    row_number() over (
      partition by c_nmrc, d_rent_dt
      order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
    ) as rn
  from rent_base
  where ods_deleted_flg not in ('1', 'Y', 'y')
),
rent_dedup as (
  select c_nmrc, d_rent_dt, n_amt_num from rent_ranked where rn = 1
),
terms_active as (
  select distinct
    cast(t.n_agr as string) as n_agr,
    cast(t.c_nmrc as string) as c_nmrc,
    cast(t.d_valid_from as date) as d_valid_from,
    cast(t.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agr_terms t
  where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and t.c_nmrc is not null
    and cast(t.d_valid_from as date) <= cast('{PERIOD_END}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{PERIOD_START}' as date))
),
agreements_active as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agreements a
  where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{PERIOD_END}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{PERIOD_START}' as date))
),
companies_active as (
  select distinct
    cast(c.n_cmp as string) as n_cmp,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
  from ods_alpha.scd1_companies c
  where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and c.c_inn is not null
)
select
  substr(cast(r.d_rent_dt as string), 1, 7) as report_month,
  c.inn_key as inn,
  cast(a.agr_id as string) as agr_id,
  count(*) as rent_rows,
  sum(r.n_amt_num) as commission_mpos
from rent_dedup r
left join terms_active t
  on t.c_nmrc = r.c_nmrc
 and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
left join agreements_active a
  on a.n_agr = t.n_agr
 and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
left join companies_active c
  on c.n_cmp = a.n_cmp_client
group by 1, 2, 3
'''

mpos_raw = fetch_imp(sql_mpos, 'MPOS mapped May-Jul')
mpos = mpos_raw.copy()
mpos['report_month'] = mpos['report_month'].astype(str).str[:7]
mpos['inn_key'] = mpos['inn'].map(normalize_inn_q1)
mpos['agr_id_key'] = mpos['agr_id'].map(normalize_agr_q1)
mpos['commission_mpos'] = pd.to_numeric(mpos['commission_mpos'], errors='coerce').round(2)
mpos_map = (
    mpos.dropna(subset=['agr_id_key'])
    .query('report_month in @FOCUS_MONTHS')
    .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
    .agg(commission_mpos=('commission_mpos', 'sum'), rent_rows=('rent_rows', 'sum'))
)
mpos_map['commission_mpos'] = mpos_map['commission_mpos'].round(2)
mpos_map['mpos_nz'] = is_nz(mpos_map['commission_mpos'])
print('MPOS keys=', f'{len(mpos_map):,}')
display(
    mpos_map.groupby('report_month', as_index=False)
    .agg(keys=('agr_id_key', 'nunique'), nz=('mpos_nz', 'sum'), sum_mpos=('commission_mpos', 'sum'))
    .sort_values('report_month')
)


## 3) OPER_DOC: `sum(c_calc_summ)` и очистка НДС 22%

В озере OPER_DOC — `ods.scd1_z_R2_IP_DOG_OPER`. Месячная комиссия с НДС = `sum(o.c_calc_summ)` на `inn + agr_id`.  
Для сверки с Excel: `commission_oper_net = round(sum(c_calc_summ) / 1.22, 2)`.


In [ ]:
sql_oper = f'''
select
  regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn,
  cast(m.id as string) as agr_id,
  cast(o.c_date_create as timestamp) as c_date_create,
  cast(o.c_calc_summ as double) as c_calc_summ
from ods.scd1_z_R2_IP_DOG_OPER o
join ods.scd1_z_r2_ip_merchants m on m.id = o.c_parent_id
join ods.scd1_z_client cl on m.c_cl_org = cl.id
where o.c_parent_class = 'R2_IP_MERCHANTS'
  and cl.class_id = 'CL_ORG'
  and cast(o.c_date_create as date) >= cast('{PERIOD_START}' as date)
  and cast(o.c_date_create as date) < cast('{PERIOD_END_EXCL}' as date)
'''
try:
    oper_raw = fetch_imp(sql_oper, 'OPER_DOC DOG_OPER May-Jul')
except Exception as exc:
    print('mixed-case fail, try lowercase:', type(exc).__name__, exc)
    sql_oper = sql_oper.replace('scd1_z_R2_IP_DOG_OPER', 'scd1_z_r2_ip_dog_oper')
    oper_raw = fetch_imp(sql_oper, 'OPER_DOC DOG_OPER May-Jul lc')

oper = oper_raw.copy()
oper['inn_key'] = oper['inn'].map(normalize_inn_q1)
oper['agr_id_key'] = oper['agr_id'].map(normalize_agr_q1)
oper['c_date_create'] = pd.to_datetime(oper['c_date_create'], errors='coerce')
oper['report_month'] = oper['c_date_create'].dt.strftime('%Y-%m')
oper['c_calc_summ'] = pd.to_numeric(oper['c_calc_summ'], errors='coerce')
oper = oper.dropna(subset=['agr_id_key'])
oper = oper.loc[oper['report_month'].isin(FOCUS_MONTHS)].copy()

oper_map = (
    oper.groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
    .agg(oper_rows=('c_calc_summ', 'size'), commission_oper_gross=('c_calc_summ', 'sum'))
)
oper_map['commission_oper_gross'] = oper_map['commission_oper_gross'].round(2)
oper_map['commission_oper_net'] = (oper_map['commission_oper_gross'] / VAT).round(2)
oper_map['oper_nz'] = is_nz(oper_map['commission_oper_net'])
print('OPER_DOC rows=', f'{len(oper):,}', '| keys=', f'{len(oper_map):,}', '| VAT=', VAT)
display(
    oper_map.groupby('report_month', as_index=False)
    .agg(
        keys=('agr_id_key', 'nunique'),
        nz=('oper_nz', 'sum'),
        sum_gross=('commission_oper_gross', 'sum'),
        sum_net=('commission_oper_net', 'sum'),
    )
    .sort_values('report_month')
)


## 4) Сводка: суммы, покрытие, точное совпадение

Одна строка на месяц и итог за май–июль. Периметр — ключи Excel.

- **Точное, %** — доля строк отчёта, где `|источник − Excel| ≤ 0.01`. Пустой источник считается 0.
- **Оба точны / только MPOS / только OPER_DOC / ни один** — только на строках отчёта с комиссией ≠ 0.
- Покрытие считается отдельно: «нет строки» и «строка есть, сумма 0» не смешиваются.


In [ ]:
keys = ['report_month', 'inn_key', 'agr_id_key']
tri = ex_all.merge(
    mpos_map[keys + ['commission_mpos', 'rent_rows']],
    on=keys,
    how='left',
)
tri = tri.merge(
    oper_map[keys + ['commission_oper_gross', 'commission_oper_net', 'oper_rows']],
    on=keys,
    how='left',
)
tri['mpos_present'] = tri['commission_mpos'].notna()
tri['oper_present'] = tri['commission_oper_gross'].notna()
tri['mpos_nz'] = is_nz(tri['commission_mpos'])
tri['oper_nz'] = is_nz(tri['commission_oper_net'])
tri['exact_mpos'] = exact_match(tri['commission_mpos'], tri['commission_excel'])
tri['exact_oper'] = exact_match(tri['commission_oper_net'], tri['commission_excel'])
tri['delta_mpos'] = tri['commission_mpos'].fillna(0).round(2) - tri['commission_excel']
tri['delta_oper'] = tri['commission_oper_net'].fillna(0).round(2) - tri['commission_excel']
tri['bucket_mpos'] = [
    classify_row(bool(ez), bool(sp), bool(sz))
    for ez, sp, sz in zip(tri['excel_nz'], tri['mpos_present'], tri['mpos_nz'])
]
tri['bucket_oper'] = [
    classify_row(bool(ez), bool(sp), bool(sz))
    for ez, sp, sz in zip(tri['excel_nz'], tri['oper_present'], tri['oper_nz'])
]
tri['three_way'] = np.select(
    [
        ~tri['excel_nz'],
        tri['exact_mpos'] & tri['exact_oper'],
        tri['exact_mpos'] & ~tri['exact_oper'],
        ~tri['exact_mpos'] & tri['exact_oper'],
    ],
    ['отчёт = 0', 'оба точны', 'только MPOS', 'только OPER_DOC'],
    default='ни один',
)

BUCKETS = [
    'оба ≠ 0',
    'оба 0 / нет строки',
    'источник ≠ 0, отчёт = 0',
    'отчёт ≠ 0, в источнике нет строки',
    'отчёт ≠ 0, источник = 0',
]


def month_row(month, frame):
    n = len(frame)
    excel_nz = frame.loc[frame['excel_nz']]
    n_nz = len(excel_nz)
    rec = {
        'report_month': month,
        'n_excel': n,
        'excel_nz': n_nz,
        'sum_excel': float(frame['commission_excel'].sum()),
        'sum_mpos': float(frame['commission_mpos'].fillna(0).sum()),
        'sum_oper_gross': float(frame['commission_oper_gross'].fillna(0).sum()),
        'sum_oper_net': float(frame['commission_oper_net'].fillna(0).sum()),
        'exact_mpos_n': int(frame['exact_mpos'].sum()),
        'exact_oper_n': int(frame['exact_oper'].sum()),
        'nz_both_exact': int((excel_nz['three_way'] == 'оба точны').sum()) if n_nz else 0,
        'nz_only_mpos': int((excel_nz['three_way'] == 'только MPOS').sum()) if n_nz else 0,
        'nz_only_oper': int((excel_nz['three_way'] == 'только OPER_DOC').sum()) if n_nz else 0,
        'nz_neither': int((excel_nz['three_way'] == 'ни один').sum()) if n_nz else 0,
    }
    rec['exact_mpos_pct'] = round(100.0 * rec['exact_mpos_n'] / n, 2) if n else np.nan
    rec['exact_oper_pct'] = round(100.0 * rec['exact_oper_n'] / n, 2) if n else np.nan
    rec['delta_sum_mpos'] = rec['sum_mpos'] - rec['sum_excel']
    rec['delta_sum_oper_net'] = rec['sum_oper_net'] - rec['sum_excel']
    for prefix, col in (('mpos', 'bucket_mpos'), ('oper', 'bucket_oper')):
        vc = frame[col].value_counts()
        rec[f'{prefix}_both_nz'] = int(vc.get('оба ≠ 0', 0))
        rec[f'{prefix}_both_zero'] = int(vc.get('оба 0 / нет строки', 0))
        rec[f'{prefix}_src_nz_excel_0'] = int(vc.get('источник ≠ 0, отчёт = 0', 0))
        rec[f'{prefix}_excel_nz_absent'] = int(vc.get('отчёт ≠ 0, в источнике нет строки', 0))
        rec[f'{prefix}_excel_nz_zero'] = int(vc.get('отчёт ≠ 0, источник = 0', 0))
    return rec


rows = [month_row(month, tri.loc[tri['report_month'] == month]) for month in FOCUS_MONTHS]
rows.append(month_row('2026-05..07', tri))
summary = pd.DataFrame(rows)

show_cols = [
    'report_month', 'n_excel', 'excel_nz',
    'sum_excel', 'sum_mpos', 'sum_oper_gross', 'sum_oper_net',
    'delta_sum_mpos', 'delta_sum_oper_net',
    'exact_mpos_n', 'exact_mpos_pct', 'exact_oper_n', 'exact_oper_pct',
    'nz_both_exact', 'nz_only_mpos', 'nz_only_oper', 'nz_neither',
    'mpos_both_nz', 'mpos_both_zero', 'mpos_src_nz_excel_0', 'mpos_excel_nz_absent', 'mpos_excel_nz_zero',
    'oper_both_nz', 'oper_both_zero', 'oper_src_nz_excel_0', 'oper_excel_nz_absent', 'oper_excel_nz_zero',
]
summary = summary[show_cols]

labels = {
    'report_month': 'Месяц',
    'n_excel': 'Строк Excel',
    'excel_nz': 'Excel ≠ 0',
    'sum_excel': 'Сумма Excel',
    'sum_mpos': 'Сумма MPOS',
    'sum_oper_gross': 'OPER_DOC с НДС',
    'sum_oper_net': 'OPER_DOC без НДС',
    'delta_sum_mpos': 'Δ суммы MPOS',
    'delta_sum_oper_net': 'Δ суммы OPER_DOC net',
    'exact_mpos_n': 'MPOS точно, шт',
    'exact_mpos_pct': 'MPOS точно, %',
    'exact_oper_n': 'OPER_DOC точно, шт',
    'exact_oper_pct': 'OPER_DOC точно, %',
    'nz_both_exact': '≠0 оба точны',
    'nz_only_mpos': '≠0 только MPOS',
    'nz_only_oper': '≠0 только OPER_DOC',
    'nz_neither': '≠0 ни один',
    'mpos_both_nz': 'MPOS оба ≠ 0',
    'mpos_both_zero': 'MPOS оба 0',
    'mpos_src_nz_excel_0': 'MPOS есть, Excel 0',
    'mpos_excel_nz_absent': 'MPOS нет строки',
    'mpos_excel_nz_zero': 'MPOS сумма 0',
    'oper_both_nz': 'OPER оба ≠ 0',
    'oper_both_zero': 'OPER оба 0',
    'oper_src_nz_excel_0': 'OPER есть, Excel 0',
    'oper_excel_nz_absent': 'OPER нет строки',
    'oper_excel_nz_zero': 'OPER сумма 0',
}
print('=== Сводка на периметре Excel ===')
display(summary.rename(columns=labels))

plot_df = summary.loc[summary['report_month'] != '2026-05..07'].copy()
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(plot_df))
width = 0.36
ax.bar(x - width / 2, plot_df['exact_mpos_pct'], width, label='MPOS_RENT')
ax.bar(x + width / 2, plot_df['exact_oper_pct'], width, label='OPER_DOC без НДС')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['report_month'])
ax.set_ylim(0, 100)
ax.set_ylabel('Точное совпадение, % строк Excel')
ax.set_title('Комиссия месяц: |источник − Excel| ≤ 0.01 ₽')
ax.legend()
fig.tight_layout()
plt.show()

total = summary.loc[summary['report_month'] == '2026-05..07'].iloc[0]
print(
    f"Май–июль точное совпадение: MPOS {total['exact_mpos_pct']}% "
    f"({int(total['exact_mpos_n']):,}/{int(total['n_excel']):,}) | "
    f"OPER_DOC без НДС {total['exact_oper_pct']}% "
    f"({int(total['exact_oper_n']):,}/{int(total['n_excel']):,})"
)
print(
    f"Суммы: Excel {total['sum_excel']:,.2f} | MPOS {total['sum_mpos']:,.2f} "
    f"(Δ {total['delta_sum_mpos']:,.2f}) | OPER_DOC net {total['sum_oper_net']:,.2f} "
    f"(Δ {total['delta_sum_oper_net']:,.2f}) | OPER_DOC gross {total['sum_oper_gross']:,.2f}"
)


## 5) Расхождения

До 30 строк на месяц, самые большие по модулю дельты. В файле — все строки периметра, где хотя бы один источник не совпал с Excel.


In [ ]:
detail_cols = [
    'report_month', 'inn_key', 'agr_id_key', 'tariff',
    'commission_excel', 'commission_mpos', 'commission_oper_gross', 'commission_oper_net',
    'delta_mpos', 'delta_oper', 'exact_mpos', 'exact_oper',
    'bucket_mpos', 'bucket_oper', 'three_way',
]
mismatch = tri.loc[~(tri['exact_mpos'] & tri['exact_oper'])].copy()
mismatch['abs_gap'] = np.maximum(mismatch['delta_mpos'].abs(), mismatch['delta_oper'].abs())
mismatch = mismatch.sort_values(['report_month', 'abs_gap'], ascending=[True, False])

print('Расхождений (не оба источника точны):', f'{len(mismatch):,}', 'из', f'{len(tri):,}')
for month in FOCUS_MONTHS:
    top = mismatch.loc[mismatch['report_month'] == month, detail_cols].head(TOP_N)
    print(f'\\n=== {month}: топ-{TOP_N} по |Δ| ===')
    display(top)

top_export = (
    mismatch.groupby('report_month', group_keys=False)
    .head(TOP_N)[detail_cols]
    .copy()
)


## 6) Выгрузка


In [ ]:
out_xlsx = OUT_DIR / 'oper_doc_vs_mpos_excel_2026_05_07.xlsx'
out_csv = OUT_DIR / 'oper_doc_vs_mpos_excel_2026_05_07_keys.csv'
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
    excel_stat.to_excel(w, sheet_name='excel_stat', index=False)
    summary.to_excel(w, sheet_name='summary', index=False)
    top_export.to_excel(w, sheet_name='top_mismatches', index=False)
tri[detail_cols].to_csv(out_csv, index=False)
print('Saved:', out_xlsx)
print('Saved:', out_csv, '| rows=', f'{len(tri):,}')
print('NOTEBOOK_REV', NOTEBOOK_REV)
print('final_df не менялся. OPER_DOC net = round(sum(c_calc_summ) / 1.22, 2)')
